In [ ]:
import os

# Place le projet à la racine du projet, pour que les chemins relatifs fonctionnent
os.chdir('..')  # remonte d'un niveau, de notebooks/ vers la racine

print("Dossier de travail :", os.getcwd())

Dossier de travail : c:\Users\progw\dashboard_energie_europe


In [23]:
import pandas as pd

url = "https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv"
df_raw = pd.read_csv(url)

print(df_raw.shape)
print(df_raw['country'].nunique())
print(df_raw['year'].min(), df_raw['year'].max())

(23377, 130)
314
1900 2025


In [24]:
# Liste des pays européens conservés pour l'analyse
pays_europe = [
    'France', 'Germany', 'Spain', 'Italy', 'Poland',
    'Sweden', 'Norway', 'Netherlands', 'Belgium', 'Austria',
    'Portugal', 'Denmark', 'Finland', 'Czechia', 'Romania',
    'Hungary', 'Switzerland', 'Greece', 'Bulgaria', 'Croatia',
    'Ireland', 'United Kingdom'
]

# Colonnes utiles uniquement
colonnes = [
    'country', 'year',
    'electricity_generation',
    'renewables_electricity',
    'solar_electricity',
    'wind_electricity',
    'hydro_electricity',
    'nuclear_electricity',
    'fossil_electricity',
    'carbon_intensity_elec',
]

# Filtrage pays + colonnes + années
df = df_raw[df_raw['country'].isin(pays_europe)][colonnes].copy()
df = df[df['year'] >= 2000].reset_index(drop=True)

# Colonnes numériques uniquement pour l'interpolation
cols_numeriques = [
    'electricity_generation', 'renewables_electricity',
    'solar_electricity', 'wind_electricity', 'hydro_electricity',
    'nuclear_electricity', 'fossil_electricity', 'carbon_intensity_elec'
]

# Interpolation par pays sur colonnes numériques uniquement
frames = []
for pays in df['country'].unique():
    subset = df[df['country'] == pays].copy()
    subset[cols_numeriques] = subset[cols_numeriques].interpolate(
        method='linear', limit_direction='both'
    )
    frames.append(subset)

df = pd.concat(frames).reset_index(drop=True)

# Colonnes calculées renouvelable et nucléaire en % de la production totale d'électricité
df['pct_renouvelable'] = (
    df['renewables_electricity'] / df['electricity_generation'] * 100
).round(1)

df['pct_nucleaire'] = (
    df['nuclear_electricity'] / df['electricity_generation'] * 100
).round(1)

# Vérifications
print(f"Lignes : {df.shape[0]}, Colonnes : {df.shape[1]}")
print(f"Pays : {df['country'].nunique()}")
print(f"Années : {df['year'].min()} à {df['year'].max()}")
print(f"\nValeurs manquantes :")
print(df.isnull().sum())
print(f"\nAperçu :")
df.head(10)

Lignes : 572, Colonnes : 12
Pays : 22
Années : 2000 à 2025

Valeurs manquantes :
country                   0
year                      0
electricity_generation    0
renewables_electricity    0
solar_electricity         0
wind_electricity          0
hydro_electricity         0
nuclear_electricity       0
fossil_electricity        0
carbon_intensity_elec     0
pct_renouvelable          0
pct_nucleaire             0
dtype: int64

Aperçu :


,country,year,electricity_generation,renewables_electricity,solar_electricity,wind_electricity,hydro_electricity,nuclear_electricity,fossil_electricity,carbon_intensity_elec,pct_renouvelable,pct_nucleaire
0,Austria,2000,59.64,43.44,0.00,0.07,41.84,0.0,16.20,203.39,72.8,0.0
1,Austria,2001,60.98,42.24,0.01,0.10,40.46,0.0,18.74,229.09,69.3,0.0
2,Austria,2002,60.63,41.90,0.01,0.14,40.23,0.0,18.73,226.13,69.1,0.0
3,Austria,2003,57.97,35.24,0.01,0.37,33.21,0.0,22.73,282.39,60.8,0.0
4,Austria,2004,61.85,39.62,0.02,0.93,36.76,0.0,22.23,259.34,64.1,0.0
5,Austria,2005,64.37,40.88,0.02,1.33,37.10,0.0,23.49,254.00,63.5,0.0
6,Austria,2006,62.21,40.66,0.02,1.75,35.66,0.0,21.55,253.01,65.4,0.0
7,Austria,2007,62.83,43.17,0.02,2.04,37.05,0.0,19.66,232.69,68.7,0.0
8,Austria,2008,64.38,44.53,0.03,2.01,38.33,0.0,19.85,220.41,69.2,0.0
9,Austria,2009,66.21,47.18,0.05,1.95,40.90,0.0,19.03,195.89,71.3,0.0


In [25]:
import duckdb
import os

# Création des dossiers si ils n'existent pas
os.makedirs('data', exist_ok=True)
os.makedirs('exports', exist_ok=True)

# Sauvegarde du dataframe nettoyé en CSV
df.to_csv('data/energy_europe_clean.csv', index=False)
print("CSV sauvegardé dans data/")

# Connexion DuckDB — crée le fichier localement
conn = duckdb.connect('data/energy_europe.duckdb')

# Table calendrier dim_date
conn.execute("""
    CREATE OR REPLACE TABLE dim_date AS
    SELECT
        year AS year_id,
        year,
        CASE
            WHEN year < 2010 THEN '2000s'
            WHEN year < 2020 THEN '2010s'
            ELSE '2020s'
        END AS decade,
        CASE
            WHEN year < 2010 THEN 'avant 2010'
            WHEN year < 2020 THEN '2010-2019'
            ELSE '2020+'
        END AS period
    FROM range(2000, 2026) t(year)
""")

# Table pays dim_country
conn.execute("""
    CREATE OR REPLACE TABLE dim_country AS
    SELECT DISTINCT
        country AS country_name,
        CASE
            WHEN country IN ('France','Spain','Italy',
                             'Belgium','Netherlands',
                             'Switzerland','Austria') THEN 'Ouest'
            WHEN country IN ('Sweden','Norway','Denmark',
                             'Finland','Ireland') THEN 'Nord'
            WHEN country IN ('Poland','Czechia','Romania',
                             'Hungary','Bulgaria',
                             'Croatia') THEN 'Est'
            WHEN country IN ('Greece','Portugal') THEN 'Sud'
            ELSE 'Autre'
        END AS region
    FROM df
""")

# Table de faits fact_energy
conn.execute("""
    CREATE OR REPLACE TABLE fact_energy AS
    SELECT
        country,
        year,
        electricity_generation,
        renewables_electricity,
        solar_electricity,
        wind_electricity,
        hydro_electricity,
        nuclear_electricity,
        fossil_electricity,
        carbon_intensity_elec,
        pct_renouvelable,
        pct_nucleaire
    FROM df
""")

# Vérification
print("\nTables créées :")
print(conn.execute("SHOW TABLES").fetchdf())
print(f"\nLignes dans fact_energy : {conn.execute('SELECT COUNT(*) FROM fact_energy').fetchone()[0]}")
print(f"Pays dans dim_country : {conn.execute('SELECT COUNT(*) FROM dim_country').fetchone()[0]}")
print(f"Années dans dim_date : {conn.execute('SELECT COUNT(*) FROM dim_date').fetchone()[0]}")

CSV sauvegardé dans data/

Tables créées :
          name
0  dim_country
1     dim_date
2  fact_energy

Lignes dans fact_energy : 572
Pays dans dim_country : 22
Années dans dim_date : 26
